# Naive Bayes Classification

Probabilistic classifier based on Bayes' theorem with the 'naive' assumption of feature independence.

## Topics Covered
1. Bayes' Theorem intuition
2. Gaussian Naive Bayes
3. Multinomial Naive Bayes (text classification)
4. Bernoulli Naive Bayes
5. Comparison of NB variants

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris, load_wine, fetch_20newsgroups
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, ConfusionMatrixDisplay
)
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings('ignore')

## Bayes' Theorem

$$P(y|X) = \frac{P(X|y) \cdot P(y)}{P(X)}$$

Where:
- $P(y|X)$ = Posterior probability (what we want)
- $P(X|y)$ = Likelihood
- $P(y)$ = Prior probability
- $P(X)$ = Evidence (constant for all classes)

## 1. Gaussian Naive Bayes (Continuous Features)

In [ ]:
iris = load_iris()
X, y = iris.data, iris.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Gaussian NB — assumes features follow Normal distribution
gnb = GaussianNB()
gnb.fit(X_train, y_train)
y_pred = gnb.predict(X_test)

print(f"Gaussian NB Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\n" + classification_report(y_test, y_pred, target_names=iris.target_names))

In [ ]:
# Visualize learned class distributions
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

for idx, (ax, feature) in enumerate(zip(axes.flatten(), iris.feature_names)):
    for cls in range(3):
        mean = gnb.theta_[cls, idx]
        var = gnb.var_[cls, idx]
        x = np.linspace(mean - 3*np.sqrt(var), mean + 3*np.sqrt(var), 100)
        pdf = (1/np.sqrt(2*np.pi*var)) * np.exp(-0.5*(x-mean)**2/var)
        ax.plot(x, pdf, label=iris.target_names[cls])
        ax.fill_between(x, pdf, alpha=0.2)
    ax.set_title(f'{feature}')
    ax.legend()

plt.suptitle('Gaussian NB: Learned Class Distributions per Feature', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=iris.target_names)
disp.plot(cmap='Greens')
plt.title('Gaussian Naive Bayes — Confusion Matrix')
plt.show()

## 2. Multinomial Naive Bayes (Text / Count Data)

In [ ]:
# Load text data
categories = ['sci.med', 'sci.space', 'rec.sport.baseball', 'talk.politics.misc']
newsgroups = fetch_20newsgroups(subset='all', categories=categories, random_state=42)

print(f"Total documents: {len(newsgroups.data)}")
print(f"Categories: {newsgroups.target_names}")

In [ ]:
# Pipeline: TF-IDF vectorization + Multinomial NB
X_train_txt, X_test_txt, y_train_txt, y_test_txt = train_test_split(
    newsgroups.data, newsgroups.target, test_size=0.25, random_state=42
)

pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=10000, stop_words='english')),
    ('nb', MultinomialNB(alpha=0.1))  # alpha = Laplace smoothing
])

pipeline.fit(X_train_txt, y_train_txt)
y_pred_txt = pipeline.predict(X_test_txt)

print(f"Multinomial NB Accuracy: {accuracy_score(y_test_txt, y_pred_txt):.4f}")
print("\n" + classification_report(y_test_txt, y_pred_txt, target_names=newsgroups.target_names))

In [ ]:
# Confusion Matrix for text classification
cm = confusion_matrix(y_test_txt, y_pred_txt)
disp = ConfusionMatrixDisplay(cm, display_labels=[c.split('.')[-1] for c in newsgroups.target_names])
disp.plot(cmap='Oranges', xticks_rotation=15)
plt.title('Multinomial NB — Text Classification')
plt.show()

## 3. Bernoulli Naive Bayes (Binary Features)

In [ ]:
# Bernoulli NB with binary vectorizer
pipeline_bnb = Pipeline([
    ('cv', CountVectorizer(binary=True, max_features=10000, stop_words='english')),
    ('bnb', BernoulliNB(alpha=1.0))
])

pipeline_bnb.fit(X_train_txt, y_train_txt)
y_pred_bnb = pipeline_bnb.predict(X_test_txt)

print(f"Bernoulli NB Accuracy: {accuracy_score(y_test_txt, y_pred_bnb):.4f}")

## 4. Comparison of NB Variants

In [ ]:
# Compare on Wine dataset (continuous features)
wine = load_wine()
X_w, y_w = wine.data, wine.target

variants = {
    'Gaussian NB': GaussianNB(),
}

print(f"{'Variant':<20} {'CV Mean':>10} {'CV Std':>10}")
print("-" * 42)
for name, model in variants.items():
    scores = cross_val_score(model, X_w, y_w, cv=10, scoring='accuracy')
    print(f"{name:<20} {scores.mean():>10.4f} {scores.std():>10.4f}")

# Compare text classifiers
text_models = {
    'Multinomial + TF-IDF': pipeline,
    'Bernoulli + Binary': pipeline_bnb
}

print(f"\n{'Text Classifier':<25} {'Accuracy':>10}")
print("-" * 37)
for name, model in text_models.items():
    acc = accuracy_score(y_test_txt, model.predict(X_test_txt))
    print(f"{name:<25} {acc:>10.4f}")

## 5. Effect of Laplace Smoothing (Alpha)

In [ ]:
alphas = [0.001, 0.01, 0.1, 0.5, 1.0, 2.0, 5.0, 10.0]
accuracies = []

for alpha in alphas:
    pipe = Pipeline([
        ('tfidf', TfidfVectorizer(max_features=10000, stop_words='english')),
        ('nb', MultinomialNB(alpha=alpha))
    ])
    pipe.fit(X_train_txt, y_train_txt)
    acc = pipe.score(X_test_txt, y_test_txt)
    accuracies.append(acc)

plt.figure(figsize=(10, 5))
plt.semilogx(alphas, accuracies, 'o-', color='steelblue', markersize=8)
plt.xlabel('Alpha (Smoothing Parameter)')
plt.ylabel('Accuracy')
plt.title('Effect of Laplace Smoothing on Multinomial NB')
plt.grid(alpha=0.3)
best_idx = np.argmax(accuracies)
plt.axvline(alphas[best_idx], color='red', linestyle='--',
            label=f'Best alpha={alphas[best_idx]} (acc={accuracies[best_idx]:.4f})')
plt.legend()
plt.tight_layout()
plt.show()

## Key Takeaways

| Variant | Best For | Feature Type |
|---------|----------|------|
| **Gaussian NB** | Continuous data (Iris, medical) | Real-valued, normally distributed |
| **Multinomial NB** | Text (TF-IDF/counts), frequencies | Non-negative integers/floats |
| **Bernoulli NB** | Binary/boolean features | 0 or 1 |

### Pros & Cons
- **Pros**: Very fast, works well with small data, good baseline, handles high dimensions
- **Cons**: Naive independence assumption, poor with correlated features